In [18]:
# Module 3 — Cleaning


import pandas as pd
import os
import numpy as np



# ============================================================
# LOAD 1 — CAMPAIGNS
# ============================================================

campaigns = pd.read_json("campaigns.json")

print("Campaigns loaded successfully")
print("Shape:", campaigns.shape)
print(campaigns.head())

# ============================================================
# LOAD 2 customers
# ============================================================


customers = pd.read_csv("customers.csv")

print("Customers loaded successfully")
print("Shape:", customers.shape)
print(customers.head())

# ============================================================
# LOAD 3 inventory
# ============================================================


inventory = pd.read_csv("inventory.csv")

print("Inventory loaded successfully")
print("Shape:", inventory.shape)
print(inventory.head())


# ============================================================
# LOAD 4 products
# ============================================================

products = pd.read_csv("products.csv")

print("Products loaded successfully")
print("Shape:", products.shape)
print(products.head())


# ============================================================
# LOAD 5 returns
# ============================================================


returns = pd.read_csv("returns.csv")

print("Returns loaded successfully")
print("Shape:", returns.shape)
print(returns.head())


# ============================================================
# LOAD 6 Store_targets_legacy
# ============================================================


store_targets_legacy = pd.read_csv("store_targets_legacy.csv")

print("Store targets legacy loaded successfully")
print("Shape:", store_targets_legacy.shape)
print(store_targets_legacy.head())

# ============================================================
# LOAD 7 STORES DATA
# ============================================================


stores = pd.read_csv("stores.csv")

print("Stores loaded successfully")
print("Shape:", stores.shape)
print(stores.head())


# ============================================================
# LOAD 8 Transactions
# ============================================================


transactions = pd.read_csv("transactions.csv")

print("Transactions loaded successfully")
print("Shape:", transactions.shape)
print(transactions.head())




Campaigns loaded successfully
Shape: (40, 9)
  campaign_id        name  channel  start_date    end_date   spend_inr  \
0      CMP401  Campaign 1    Email  2024-11-30  2025-01-11   200724.57   
1      CMP402  Campaign 2  Display  2024-10-26  2024-11-05  1097250.03   
2      CMP403  Campaign 3   Search  2024-11-07  2024-12-09   141428.85   
3      CMP404  Campaign 4      SMS  2024-03-16  2024-03-24  1769200.56   
4      CMP405  Campaign 5  Display  2023-07-01  2023-08-27   229130.74   

   impressions  clicks target_category  
0      2328679   21852          Beauty  
1      2076813   87955          Beauty  
2      1189562   81707     Electronics  
3      1279210   53014             All  
4      2992457   46695     Electronics  
Customers loaded successfully
Shape: (3022, 8)
  CustomerID CustomerName Gender AgeBand        City LoyaltyTier    JoinDate  \
0    CU10001  Vihaan Khan      M   26-35     Chennai      Silver  2023-11-03   
1    CU10002   Aman Reddy      F   46-60      Mumbai     

In [19]:
# ============================================================
# 1. CLEANING LOG
# ============================================================

cleaning_log = []


def log_cleaning(
    table,
    column,
    issue,
    rows_affected,
    action,
    justification
):

    cleaning_log.append({
        "Table": table,
        "Column": column,
        "Issue": issue,
        "Rows Affected": rows_affected,
        "Action": action,
        "Justification": justification
    })


# ============================================================
# 2. HELPER FUNCTION — STANDARDISE TEXT
# ============================================================

def standardise_text(df, table_name, column):

    if column not in df.columns:
        return

    before = df[column].copy()

    # Convert non-null values to string
    df[column] = df[column].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

    # Replace multiple spaces with single space
    df[column] = df[column].apply(
        lambda x: " ".join(x.split()) if isinstance(x, str) else x
    )

    # Count changed rows
    changed = (
        before.fillna("__NULL__").astype(str)
        != df[column].fillna("__NULL__").astype(str)
    ).sum()

    if changed > 0:

        log_cleaning(
            table_name,
            column,
            "Leading/trailing or repeated whitespace",
            changed,
            "Trimmed whitespace and standardised spacing",
            "Text fields should have consistent formatting so that "
            "matching, grouping and joins are reliable."
        )


In [20]:

# ============================================================
# 3. CUSTOMERS
# ============================================================

print("\n" + "=" * 80)
print("CLEANING — CUSTOMERS")
print("=" * 80)


# CustomerID
# Keep identifiers unchanged except whitespace.

standardise_text(customers, "Customers", "CustomerID")


# CustomerName
standardise_text(customers, "Customers", "CustomerName")


# Gender
standardise_text(customers, "Customers", "Gender")

if "Gender" in customers.columns:

    before = customers["Gender"].copy()

    customers["Gender"] = (
        customers["Gender"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    changed = (
        before.fillna("__NULL__").astype(str)
        != customers["Gender"].fillna("__NULL__").astype(str)
    ).sum()

    if changed > 0:

        log_cleaning(
            "Customers",
            "Gender",
            "Inconsistent text case",
            changed,
            "Standardised Gender to title case",
            "Consistent categorical labels are required for "
            "accurate grouping and reporting."
        )


# AgeBand
standardise_text(customers, "Customers", "AgeBand")


# City
if "City" in customers.columns:

    before = customers["City"].copy()

    customers["City"] = (
        customers["City"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    changed = (
        before.fillna("__NULL__").astype(str)
        != customers["City"].fillna("__NULL__").astype(str)
    ).sum()

    if changed > 0:

        log_cleaning(
            "Customers",
            "City",
            "Inconsistent city formatting",
            changed,
            "Trimmed whitespace and standardised city names",
            "City values must use consistent labels for geographic analysis."
        )


# LoyaltyTier
if "LoyaltyTier" in customers.columns:

    before = customers["LoyaltyTier"].copy()

    customers["LoyaltyTier"] = (
        customers["LoyaltyTier"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    changed = (
        before.fillna("__NULL__").astype(str)
        != customers["LoyaltyTier"].fillna("__NULL__").astype(str)
    ).sum()

    if changed > 0:

        log_cleaning(
            "Customers",
            "LoyaltyTier",
            "Inconsistent categorical formatting",
            changed,
            "Standardised LoyaltyTier labels",
            "Consistent categories are required for reliable customer segmentation."
        )


# JoinDate
if "JoinDate" in customers.columns:

    before_invalid = customers["JoinDate"].notna().sum()

    customers["JoinDate"] = pd.to_datetime(
        customers["JoinDate"],
        errors="coerce"
    )

    after_invalid = customers["JoinDate"].isna().sum()

    if after_invalid > 0:

        log_cleaning(
            "Customers",
            "JoinDate",
            "Invalid or missing dates",
            after_invalid,
            "Converted valid values to datetime; invalid values retained as NaT",
            "Invalid dates cannot be used reliably for customer tenure analysis."
        )


# EmailOptIn
standardise_text(customers, "Customers", "EmailOptIn")


CLEANING — CUSTOMERS


In [21]:

# ============================================================
# 4. INVENTORY
# ============================================================

print("\n" + "=" * 80)
print("CLEANING — INVENTORY")
print("=" * 80)


standardise_text(inventory, "Inventory", "StoreID")
standardise_text(inventory, "Inventory", "ProductID")


# SnapshotMonth
if "SnapshotMonth" in inventory.columns:

    inventory["SnapshotMonth"] = pd.to_datetime(
        inventory["SnapshotMonth"],
        errors="coerce"
    )

    log_cleaning(
        "Inventory",
        "SnapshotMonth",
        "Date/period formatting",
        len(inventory),
        "Converted SnapshotMonth to datetime",
        "Monthly inventory snapshots should use a consistent date type."
    )


# OnHandUnits
if "OnHandUnits" in inventory.columns:

    negative_count = (inventory["OnHandUnits"] < 0).sum()

    if negative_count > 0:

        log_cleaning(
            "Inventory",
            "OnHandUnits",
            "Negative inventory quantity",
            negative_count,
            "Flagged for business review; values were not automatically replaced",
            "Negative stock may represent a legitimate adjustment or data issue. "
            "It should not be replaced without business evidence."
        )


# ReorderLevel
if "ReorderLevel" in inventory.columns:

    negative_count = (inventory["ReorderLevel"] < 0).sum()

    if negative_count > 0:

        log_cleaning(
            "Inventory",
            "ReorderLevel",
            "Negative reorder level",
            negative_count,
            "Flagged for business review; values were not automatically replaced",
            "A reorder threshold should normally be non-negative, but the correct "
            "replacement cannot be inferred from the dataset."
        )


# StockoutFlag
if "StockoutFlag" in inventory.columns:

    invalid_flag = ~inventory["StockoutFlag"].isin([0, 1])
    invalid_count = invalid_flag.sum()

    if invalid_count > 0:

        log_cleaning(
            "Inventory",
            "StockoutFlag",
            "Invalid flag values",
            invalid_count,
            "Flagged for review",
            "StockoutFlag should contain binary values only."
        )



CLEANING — INVENTORY


In [22]:

# ============================================================
# 5. PRODUCTS
# ============================================================

print("\n" + "=" * 80)
print("CLEANING — PRODUCTS")
print("=" * 80)


standardise_text(products, "Products", "ProductID")
standardise_text(products, "Products", "ProductName")
standardise_text(products, "Products", "Category")
standardise_text(products, "Products", "SubCategory")
standardise_text(products, "Products", "Brand")


# LaunchDate
if "LaunchDate" in products.columns:

    products["LaunchDate"] = pd.to_datetime(
        products["LaunchDate"],
        errors="coerce"
    )

    log_cleaning(
        "Products",
        "LaunchDate",
        "Date formatting",
        len(products),
        "Converted LaunchDate to datetime",
        "A consistent date type is required for product age and launch analysis."
    )


# UnitCost
if "UnitCost" in products.columns:

    negative_count = (products["UnitCost"] < 0).sum()

    if negative_count > 0:

        log_cleaning(
            "Products",
            "UnitCost",
            "Negative cost",
            negative_count,
            "Flagged for review",
            "Product cost cannot normally be negative; automatic replacement "
            "would require a business rule."
        )


# MRP
if "MRP" in products.columns:

    negative_count = (products["MRP"] < 0).sum()

    if negative_count > 0:

        log_cleaning(
            "Products",
            "MRP",
            "Negative selling price",
            negative_count,
            "Flagged for review",
            "MRP should not be negative and should be investigated before replacement."
        )


CLEANING — PRODUCTS


In [23]:

# ============================================================
# 6. RETURNS
# ============================================================

print("\n" + "=" * 80)
print("CLEANING — RETURNS")
print("=" * 80)


standardise_text(returns, "Returns", "ReturnID")
standardise_text(returns, "Returns", "TransactionID")
standardise_text(returns, "Returns", "Reason")
standardise_text(returns, "Returns", "Condition")


# ReturnDate
if "ReturnDate" in returns.columns:

    returns["ReturnDate"] = pd.to_datetime(
        returns["ReturnDate"],
        errors="coerce"
    )

    log_cleaning(
        "Returns",
        "ReturnDate",
        "Date formatting",
        len(returns),
        "Converted ReturnDate to datetime",
        "Return dates must use a consistent date type for return trend analysis."
    )


# RefundAmount
if "RefundAmount" in returns.columns:

    negative_count = (returns["RefundAmount"] < 0).sum()

    if negative_count > 0:

        log_cleaning(
            "Returns",
            "RefundAmount",
            "Negative refund amount",
            negative_count,
            "Flagged for review",
            "Negative refunds require business confirmation and should not "
            "be replaced automatically."
        )


CLEANING — RETURNS


In [24]:

# ============================================================
# 7. STORE TARGETS LEGACY
# ============================================================

print("\n" + "=" * 80)
print("CLEANING — STORE TARGETS LEGACY")
print("=" * 80)


standardise_text(
    store_targets_legacy,
    "Store Targets Legacy",
    "store_code"
)


standardise_text(
    store_targets_legacy,
    "Store Targets Legacy",
    "period"
)

if "period" in store_targets_legacy.columns:

    store_targets_legacy["period"] = pd.to_datetime(
        store_targets_legacy["period"],
        format="%b-%Y",
        errors="coerce"
    )

    log_cleaning(
        "Store Targets Legacy",
        "period",
        "Period formatting",
        len(store_targets_legacy),
        "Converted Jan-YYYY values to datetime",
        "Consistent period dates are required for target-versus-actual analysis."
    )

# Sales target
if "sales_target_inr" in store_targets_legacy.columns:

    negative_count = (
        store_targets_legacy["sales_target_inr"] < 0
    ).sum()

    if negative_count > 0:

        log_cleaning(
            "Store Targets Legacy",
            "sales_target_inr",
            "Negative sales target",
            negative_count,
            "Flagged for review",
            "Sales targets should not normally be negative."
        )


# Footfall target
if "footfall_target" in store_targets_legacy.columns:

    negative_count = (
        store_targets_legacy["footfall_target"] < 0
    ).sum()

    if negative_count > 0:

        log_cleaning(
            "Store Targets Legacy",
            "footfall_target",
            "Negative footfall target",
            negative_count,
            "Flagged for review",
            "Footfall targets should not normally be negative."
        )


CLEANING — STORE TARGETS LEGACY


In [25]:

# ============================================================
# 8. STORES
# ============================================================

print("\n" + "=" * 80)
print("CLEANING — STORES")
print("=" * 80)


standardise_text(stores, "Stores", "StoreID")
standardise_text(stores, "Stores", "StoreName")
standardise_text(stores, "Stores", "City")
standardise_text(stores, "Stores", "State")
standardise_text(stores, "Stores", "Format")
standardise_text(stores, "Stores", "Region")


# OpenDate
if "OpenDate" in stores.columns:

    stores["OpenDate"] = pd.to_datetime(
        stores["OpenDate"],
        errors="coerce"
    )

    log_cleaning(
        "Stores",
        "OpenDate",
        "Date formatting",
        len(stores),
        "Converted OpenDate to datetime",
        "Store opening dates need a consistent date type for store age analysis."
    )


# SqFt
if "SqFt" in stores.columns:

    negative_count = (stores["SqFt"] < 0).sum()

    if negative_count > 0:

        log_cleaning(
            "Stores",
            "SqFt",
            "Negative store area",
            negative_count,
            "Flagged for review",
            "Store area cannot normally be negative."
        )


CLEANING — STORES


In [26]:
#============================================================
# 9. TRANSACTIONS
# ============================================================

print("\n" + "=" * 80)
print("CLEANING — TRANSACTIONS")
print("=" * 80)


standardise_text(transactions, "Transactions", "TransactionID")
standardise_text(transactions, "Transactions", "CustomerID")
standardise_text(transactions, "Transactions", "StoreID")
standardise_text(transactions, "Transactions", "ProductID")
standardise_text(transactions, "Transactions", "Channel")
standardise_text(transactions, "Transactions", "PaymentMode")


# OrderDate
if "OrderDate" in transactions.columns:

    transactions["OrderDate"] = pd.to_datetime(
        transactions["OrderDate"],
        errors="coerce"
    )

    log_cleaning(
        "Transactions",
        "OrderDate",
        "Date formatting",
        len(transactions),
        "Converted OrderDate to datetime",
        "Transaction dates must use a consistent date type for sales analysis."
    )


# Quantity
if "Quantity" in transactions.columns:

    negative_count = (
        transactions["Quantity"] < 0
    ).sum()

    if negative_count > 0:

        log_cleaning(
            "Transactions",
            "Quantity",
            "Negative quantity",
            negative_count,
            "Flagged for review",
            "Negative quantities may represent returns or adjustments and "
            "should not be overwritten without business rules."
        )


# Discount
if "Discount" in transactions.columns:

    invalid_discount = (
        (transactions["Discount"] < 0)
        | (transactions["Discount"] > 1)
    )

    invalid_count = invalid_discount.sum()

    if invalid_count > 0:

        log_cleaning(
            "Transactions",
            "Discount",
            "Discount outside expected range",
            invalid_count,
            "Flagged for review",
            "Discount values should follow a consistent business definition."
        )


# LineAmount
if "LineAmount" in transactions.columns:

    negative_count = (
        transactions["LineAmount"] < 0
    ).sum()

    if negative_count > 0:

        log_cleaning(
            "Transactions",
            "LineAmount",
            "Negative transaction amount",
            negative_count,
            "Flagged for review",
            "Negative amounts may represent refunds or adjustments and "
            "should not be automatically deleted."
        )


CLEANING — TRANSACTIONS


In [27]:

# ============================================================
# 10. CAMPAIGNS
# ============================================================

print("\n" + "=" * 80)
print("CLEANING — CAMPAIGNS")
print("=" * 80)


standardise_text(campaigns, "Campaigns", "campaign_id")
standardise_text(campaigns, "Campaigns", "name")
standardise_text(campaigns, "Campaigns", "channel")
standardise_text(campaigns, "Campaigns", "target_category")


# Start date
if "start_date" in campaigns.columns:

    campaigns["start_date"] = pd.to_datetime(
        campaigns["start_date"],
        errors="coerce"
    )

    log_cleaning(
        "Campaigns",
        "start_date",
        "Date formatting",
        len(campaigns),
        "Converted start_date to datetime",
        "Campaign dates must use a consistent date type for campaign analysis."
    )


# End date
if "end_date" in campaigns.columns:

    campaigns["end_date"] = pd.to_datetime(
        campaigns["end_date"],
        errors="coerce"
    )

    log_cleaning(
        "Campaigns",
        "end_date",
        "Date formatting",
        len(campaigns),
        "Converted end_date to datetime",
        "Campaign dates must use a consistent date type for duration analysis."
    )


# Spend
if "spend_inr" in campaigns.columns:

    negative_count = (
        campaigns["spend_inr"] < 0
    ).sum()

    if negative_count > 0:

        log_cleaning(
            "Campaigns",
            "spend_inr",
            "Negative campaign spend",
            negative_count,
            "Flagged for review",
            "Campaign spend should not normally be negative."
        )


# Impressions
if "impressions" in campaigns.columns:

    negative_count = (
        campaigns["impressions"] < 0
    ).sum()

    if negative_count > 0:

        log_cleaning(
            "Campaigns",
            "impressions",
            "Negative impressions",
            negative_count,
            "Flagged for review",
            "Impressions represent counts and should not be negative."
        )


# Clicks
if "clicks" in campaigns.columns:

    negative_count = (
        campaigns["clicks"] < 0
    ).sum()

    if negative_count > 0:

        log_cleaning(
            "Campaigns",
            "clicks",
            "Negative clicks",
            negative_count,
            "Flagged for review",
            "Clicks represent counts and should not be negative."
        )


CLEANING — CAMPAIGNS


In [34]:
# ============================================================
# 11. REFERENTIAL INTEGRITY CHECK
# ============================================================

print("\n" + "=" * 80)
print("11. REFERENTIAL INTEGRITY CHECK")
print("=" * 80)

referential_issues = []


# ------------------------------------------------------------
# Helper function: check child IDs against parent IDs
# ------------------------------------------------------------

def check_referential_integrity(
    child_df,
    child_column,
    parent_df,
    parent_column,
    child_table,
    parent_table,
    allow_values=None
):

    if allow_values is None:
        allow_values = []

    # Normalise IDs for comparison without changing source columns
    child_ids = (
        child_df[child_column]
        .astype("string")
        .str.strip()
    )

    parent_ids = set(
        parent_df[parent_column]
        .dropna()
        .astype("string")
        .str.strip()
    )

    # Missing foreign keys
    missing_mask = child_df[child_column].isna()

    # Allowed special values, such as ONLINE
    allowed_mask = child_ids.isin(allow_values)

    # Orphan foreign keys
    orphan_mask = (
        child_ids.notna()
        & ~child_ids.isin(parent_ids)
        & ~allowed_mask
    )

    missing_count = int(missing_mask.sum())
    orphan_count = int(orphan_mask.sum())

    print(f"\n{child_table} -> {parent_table}")
    print(f"Missing {child_column}: {missing_count}")
    print(f"Orphan {child_column}: {orphan_count}")

    # Record missing foreign keys
    if missing_count > 0:

        log_cleaning(
            child_table,
            child_column,
            "Missing foreign key",
            missing_count,
            "Retained and flagged for review",
            "The correct ID cannot be safely inferred."
        )

        for row_index in child_df.index[missing_mask]:

            referential_issues.append({
                "Table": child_table,
                "Column": child_column,
                "RowIndex": row_index,
                "InvalidID": None,
                "Issue": "Missing foreign key",
                "ParentTable": parent_table
            })

    # Record orphan foreign keys
    if orphan_count > 0:

        log_cleaning(
            child_table,
            child_column,
            "Referential mismatch",
            orphan_count,
            "Retained and flagged for review",
            f"ID does not exist in {parent_table}."
        )

        for row_index in child_df.index[orphan_mask]:

            referential_issues.append({
                "Table": child_table,
                "Column": child_column,
                "RowIndex": row_index,
                "InvalidID": child_ids.loc[row_index],
                "Issue": "Referential mismatch",
                "ParentTable": parent_table
            })

    return {
        "missing_count": missing_count,
        "orphan_count": orphan_count
    }


# ------------------------------------------------------------
# 11.1 Transactions.CustomerID -> Customers.CustomerID
# ------------------------------------------------------------

check_referential_integrity(
    transactions,
    "CustomerID",
    customers,
    "CustomerID",
    "Transactions",
    "Customers"
)


# ------------------------------------------------------------
# 11.2 Transactions.StoreID -> Stores.StoreID
# ------------------------------------------------------------

check_referential_integrity(
    transactions,
    "StoreID",
    stores,
    "StoreID",
    "Transactions",
    "Stores",
    allow_values=["ONLINE"]
)


# ------------------------------------------------------------
# 11.3 Transactions.ProductID -> Products.ProductID
# ------------------------------------------------------------

check_referential_integrity(
    transactions,
    "ProductID",
    products,
    "ProductID",
    "Transactions",
    "Products"
)


# ------------------------------------------------------------
# 11.4 Inventory.StoreID -> Stores.StoreID
# ------------------------------------------------------------

check_referential_integrity(
    inventory,
    "StoreID",
    stores,
    "StoreID",
    "Inventory",
    "Stores"
)


# ------------------------------------------------------------
# 11.5 Inventory.ProductID -> Products.ProductID
# ------------------------------------------------------------

check_referential_integrity(
    inventory,
    "ProductID",
    products,
    "ProductID",
    "Inventory",
    "Products"
)


# ------------------------------------------------------------
# 11.6 Returns.TransactionID -> Transactions.TransactionID
# ------------------------------------------------------------

check_referential_integrity(
    returns,
    "TransactionID",
    transactions,
    "TransactionID",
    "Returns",
    "Transactions"
)


# ------------------------------------------------------------
# 11.7 Store Targets.store_code -> Stores.StoreID
# ------------------------------------------------------------

check_referential_integrity(
    store_targets_legacy,
    "store_code",
    stores,
    "StoreID",
    "Store Targets Legacy",
    "Stores"
)


11. REFERENTIAL INTEGRITY CHECK

Transactions -> Customers
Missing CustomerID: 16874
Orphan CustomerID: 0

Transactions -> Stores
Missing StoreID: 0
Orphan StoreID: 0

Transactions -> Products
Missing ProductID: 0
Orphan ProductID: 45

Inventory -> Stores
Missing StoreID: 0
Orphan StoreID: 0

Inventory -> Products
Missing ProductID: 0
Orphan ProductID: 0

Returns -> Transactions
Missing TransactionID: 0
Orphan TransactionID: 0

Store Targets Legacy -> Stores
Missing store_code: 0
Orphan store_code: 0


{'missing_count': 0, 'orphan_count': 0}

In [36]:
# ============================================================
# 12. SAVE REFERENTIAL ISSUES REPORT
# ============================================================

os.makedirs("cleaned_data", exist_ok=True)

referential_issues_df = pd.DataFrame(
    referential_issues,
    columns=[
        "Table",
        "Column",
        "RowIndex",
        "InvalidID",
        "Issue",
        "ParentTable"
    ]
)

referential_issues_df.to_csv(
    "cleaned_data/referential_issues.csv",
    index=False
)

print("\nReferential issues saved.")


Referential issues saved.


In [30]:
# ============================================================
# 13. SAVE DATA CLEANING LOG
# ============================================================

cleaning_log_df = pd.DataFrame(
    cleaning_log,
    columns=[
        "Table",
        "Column",
        "Issue",
        "Rows Affected",
        "Action",
        "Justification"
    ]
)

cleaning_log_df.to_csv(
    "cleaned_data/data_cleaning_log.csv",
    index=False
)

print("Cleaning log saved.")

Cleaning log saved.


In [31]:

# ============================================================
# 14. EXPORT ALL EIGHT CLEANED TABLES
# ============================================================

cleaned_tables = {
    "customers": customers,
    "inventory": inventory,
    "products": products,
    "returns": returns,
    "store_targets_legacy": store_targets_legacy,
    "stores": stores,
    "transactions": transactions,
    "campaigns": campaigns
}

for table_name, dataframe in cleaned_tables.items():

    output_path = os.path.join(
        "cleaned_data",
        f"cleaned_{table_name}.csv"
    )

    dataframe.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved {table_name}: "
        f"{dataframe.shape} -> {output_path}"
    )

Saved customers: (3022, 8) -> cleaned_data/cleaned_customers.csv
Saved inventory: (25920, 6) -> cleaned_data/cleaned_inventory.csv
Saved products: (528, 8) -> cleaned_data/cleaned_products.csv
Saved returns: (3415, 6) -> cleaned_data/cleaned_returns.csv
Saved store_targets_legacy: (432, 4) -> cleaned_data/cleaned_store_targets_legacy.csv
Saved stores: (18, 8) -> cleaned_data/cleaned_stores.csv
Saved transactions: (60577, 10) -> cleaned_data/cleaned_transactions.csv
Saved campaigns: (40, 9) -> cleaned_data/cleaned_campaigns.csv


In [37]:
# ============================================================
# 15. FINAL DATA QUALITY VALIDATION
# ============================================================

validation_results = []

for table_name, dataframe in cleaned_tables.items():

    print("\n" + "=" * 70)
    print(f"FINAL VALIDATION: {table_name}")
    print("=" * 70)

    print("Shape:", dataframe.shape)

    print("\nMissing values:")
    print(dataframe.isna().sum())

    duplicate_count = int(dataframe.duplicated().sum())

    print("\nDuplicate full rows:", duplicate_count)

    print("\nData types:")
    print(dataframe.dtypes)

    # Count remaining text whitespace issues
    text_columns = dataframe.select_dtypes(
        include=["object", "string"]
    ).columns

    whitespace_issues = 0

    for column in text_columns:

        values = dataframe[column].dropna().astype(str)

        whitespace_issues += int(
            (
                values.str.strip() != values
            ).sum()
        )

    # Count remaining invalid dates in date-like columns
    date_columns = [
        column
        for column in dataframe.columns
        if (
            "date" in column.lower()
            or "month" in column.lower()
            or "period" in column.lower()
        )
        and not pd.api.types.is_datetime64_any_dtype(
            dataframe[column]
        )
    ]

    invalid_dates = 0

    for column in date_columns:

        converted_dates = pd.to_datetime(
            dataframe[column],
            errors="coerce"
        )

        invalid_dates += int(
            (
                dataframe[column].notna()
                & converted_dates.isna()
            ).sum()
        )

    validation_results.append({
        "Table": table_name,
        "Rows": dataframe.shape[0],
        "Columns": dataframe.shape[1],
        "MissingCells": int(
            dataframe.isna().sum().sum()
        ),
        "DuplicateFullRows": duplicate_count,
        "WhitespaceIssues": whitespace_issues,
        "InvalidDates": invalid_dates
    })


# Save validation summary
validation_df = pd.DataFrame(validation_results)

validation_df.to_csv(
    "cleaned_data/data_validation_summary.csv",
    index=False
)

print("\nValidation summary saved.")


FINAL VALIDATION: customers
Shape: (3022, 8)

Missing values:
CustomerID       0
CustomerName     0
Gender           0
AgeBand         25
City             0
LoyaltyTier     38
JoinDate         0
EmailOptIn       0
dtype: int64

Duplicate full rows: 22

Data types:
CustomerID              object
CustomerName            object
Gender          string[python]
AgeBand                 object
City            string[python]
LoyaltyTier     string[python]
JoinDate        datetime64[ns]
EmailOptIn              object
dtype: object

FINAL VALIDATION: inventory
Shape: (25920, 6)

Missing values:
SnapshotMonth      0
StoreID            0
ProductID          0
OnHandUnits      120
ReorderLevel       0
StockoutFlag       0
dtype: int64

Duplicate full rows: 0

Data types:
SnapshotMonth    datetime64[ns]
StoreID                  object
ProductID                object
OnHandUnits             float64
ReorderLevel              int64
StockoutFlag              int64
dtype: object

FINAL VALIDATION: product

In [33]:
#============================================================
# 16. MODULE 3 COMPLETION SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("MODULE 3 — DATA CLEANING COMPLETED")
print("=" * 80)

print("Tables exported:", len(cleaned_tables))
print("Cleaning log entries:", len(cleaning_log))
print(
    "Referential issue records:",
    len(referential_issues_df)
)

print("\nOutput folder: cleaned")

for table_name in cleaned_tables:

    print(f"- cleaned_{table_name}.csv")

print("- data_cleaning_log.csv")
print("- referential_issues.csv")
print("- data_validation_summary.csv")

print("\nReview remaining issues before proceeding to analysis.")


MODULE 3 — DATA CLEANING COMPLETED
Tables exported: 8
Cleaning log entries: 12
Referential issue records: 16919

Output folder: cleaned_data/
- cleaned_customers.csv
- cleaned_inventory.csv
- cleaned_products.csv
- cleaned_returns.csv
- cleaned_store_targets_legacy.csv
- cleaned_stores.csv
- cleaned_transactions.csv
- cleaned_campaigns.csv
- data_cleaning_log.csv
- referential_issues.csv
- data_validation_summary.csv

Review remaining issues before proceeding to analysis.


In [ ]:
# ============================================================
# 11. REFERENTIAL INTEGRITY
# ============================================================

print("\n" + "=" * 80)
print("REFERENTIAL INTEGRITY CHECK")
print("=" * 80)


# ------------------------------------------------------------
# Transactions.CustomerID -> Customers.CustomerID
# ------------------------------------------------------------

customer_ids = set(
    customers["CustomerID"]
    .dropna()
    .astype(str)
    .str.strip()
)

transaction_customer_ids = (
    transactions["CustomerID"]
    .dropna()
    .astype(str)
    .str.strip()
)

orphan_transaction_customers = (
    ~transaction_customer_ids.isin(customer_ids)
)

orphan_customer_count = orphan_transaction_customers.sum()

missing_customer_count = transactions["CustomerID"].isna().sum()

print(
    f"Transactions -> Customers orphan IDs: "
    f"{orphan_customer_count}"
)

print(
    f"Transactions with missing CustomerID: "
    f"{missing_customer_count}"
)

if orphan_customer_count > 0:

    log_cleaning(
        "Transactions",
        "CustomerID",
        "Referential mismatch",
        orphan_customer_count,
        "Flagged orphan CustomerID values for review",
        "Every non-missing transaction CustomerID should exist in Customers."
    )

if missing_customer_count > 0:

    log_cleaning(
        "Transactions",
        "CustomerID",
        "Missing CustomerID",
        missing_customer_count,
        "Retained transactions and flagged missing CustomerID values for review",
        "CustomerID is required to link transactions to Customers. Missing IDs cannot be safely inferred."
    )


# ------------------------------------------------------------
# Transactions.StoreID -> Stores.StoreID
# ------------------------------------------------------------

store_ids = set(
    stores["StoreID"]
    .dropna()
    .astype(str)
    .str.strip()
)

transaction_store_ids = (
    transactions["StoreID"]
    .dropna()
    .astype(str)
    .str.strip()
)

# ONLINE is a valid sales channel, not a physical store
online_count = (
    transaction_store_ids.eq("ONLINE").sum()
)

physical_store_ids = (
    transaction_store_ids[
        transaction_store_ids.ne("ONLINE")
    ]
)

orphan_transaction_stores = (
    ~physical_store_ids.isin(store_ids)
)

orphan_store_count = orphan_transaction_stores.sum()

print(
    f"Transactions with StoreID = ONLINE: "
    f"{online_count}"
)

print(
    f"Transactions -> Stores physical-store mismatches: "
    f"{orphan_store_count}"
)

if orphan_store_count > 0:

    log_cleaning(
        "Transactions",
        "StoreID",
        "Referential mismatch",
        orphan_store_count,
        "Flagged orphan physical StoreID values for review",
        "Every physical StoreID should exist in the Stores master table."
    )


# ------------------------------------------------------------
# Transactions.ProductID -> Products.ProductID
# ------------------------------------------------------------

product_ids = set(
    products["ProductID"]
    .dropna()
    .astype(str)
    .str.strip()
)

transaction_product_ids = (
    transactions["ProductID"]
    .dropna()
    .astype(str)
    .str.strip()
)

orphan_transaction_products = (
    ~transaction_product_ids.isin(product_ids)
)

orphan_product_count = orphan_transaction_products.sum()

print(
    f"Transactions -> Products mismatches: "
    f"{orphan_product_count}"
)

if orphan_product_count > 0:

    log_cleaning(
        "Transactions",
        "ProductID",
        "Referential mismatch",
        orphan_product_count,
        "Flagged orphan ProductID values for review",
        "Transaction ProductIDs P9000-P9044 are not present in the Products master table and cannot be safely mapped."
    )


REFERENTIAL INTEGRITY CHECK
Transactions -> Customers orphan IDs: 0
Transactions with missing CustomerID: 16874
Transactions with StoreID = ONLINE: 27969
Transactions -> Stores physical-store mismatches: 0
Transactions -> Products mismatches: 45


In [ ]:
# ============================================================
# REFERENTIAL INTEGRITY SUMMARY
# ============================================================
#
# CustomerID:
# - 0 orphan non-missing CustomerIDs
# - 16,874 transactions have missing CustomerID
# - Missing IDs were retained and flagged because they cannot
#   be safely inferred.
#
# StoreID:
# - 27,969 transactions use ONLINE as StoreID.
# - ONLINE is treated as a valid non-physical sales channel.
# - 0 physical StoreID mismatches remain.
#
# ProductID:
# - 45 transactions contain ProductIDs P9000-P9044
#   that are absent from the Products master table.
# - These records were retained and flagged for review.
#
# All actions are recorded in data_cleaning_log.csv.

In [ ]:

# ------------------------------------------------------------
# Inventory.StoreID -> Stores.StoreID
# ------------------------------------------------------------

inventory_store_ids = (
    inventory["StoreID"]
    .dropna()
    .astype(str)
    .str.strip()
)

orphan_inventory_stores = (
    ~inventory_store_ids.isin(store_ids)
)

orphan_count = orphan_inventory_stores.sum()

print(
    f"Inventory -> Stores mismatches: {orphan_count}"
)

if orphan_count > 0:

    log_cleaning(
        "Inventory",
        "StoreID",
        "Referential mismatch",
        orphan_count,
        "Flagged orphan StoreID values for review",
        "Every inventory StoreID should correspond to a valid store."
    )


# ------------------------------------------------------------
# Inventory.ProductID -> Products.ProductID
# ------------------------------------------------------------

inventory_product_ids = (
    inventory["ProductID"]
    .dropna()
    .astype(str)
    .str.strip()
)

orphan_inventory_products = (
    ~inventory_product_ids.isin(product_ids)
)

orphan_count = orphan_inventory_products.sum()

print(
    f"Inventory -> Products mismatches: {orphan_count}"
)

if orphan_count > 0:

    log_cleaning(
        "Inventory",
        "ProductID",
        "Referential mismatch",
        orphan_count,
        "Flagged orphan ProductID values for review",
        "Every inventory ProductID should correspond to a valid product."
    )




# ------------------------------------------------------------
# Returns.TransactionID -> Transactions.TransactionID
# ------------------------------------------------------------

transaction_ids = set(
    transactions["TransactionID"]
    .dropna()
    .astype(str)
    .str.strip()
)

return_transaction_ids = (
    returns["TransactionID"]
    .dropna()
    .astype(str)
    .str.strip()
)

orphan_return_transactions = (
    ~return_transaction_ids.isin(transaction_ids)
)

orphan_count = orphan_return_transactions.sum()

print(
    f"Returns -> Transactions mismatches: {orphan_count}"
)

if orphan_count > 0:

    log_cleaning(
        "Returns",
        "TransactionID",
        "Referential mismatch",
        orphan_count,
        "Flagged orphan TransactionID values for review",
        "Every return should reference an existing transaction."
    )


# ------------------------------------------------------------
# Store Targets -> Stores
# ------------------------------------------------------------

target_store_ids = (
    store_targets_legacy["store_code"]
    .dropna()
    .astype(str)
    .str.strip()
)

orphan_target_stores = (
    ~target_store_ids.isin(store_ids)
)

orphan_count = orphan_target_stores.sum()

print(
    f"Store Targets -> Stores mismatches: {orphan_count}"
)

if orphan_count > 0:

    log_cleaning(
        "Store Targets Legacy",
        "store_code",
        "Referential mismatch",
        orphan_count,
        "Flagged orphan store codes for review",
        "Every target record should reference an existing store."
    )

Inventory -> Stores mismatches: 0
Inventory -> Products mismatches: 0
Returns -> Transactions mismatches: 0
Store Targets -> Stores mismatches: 0


In [ ]:
# ============================================================
# 12. SAVE CLEANING LOG
# ============================================================

cleaning_log_df = pd.DataFrame(cleaning_log)

cleaning_log_df.to_csv(
    "data_cleaning_log.csv",
    index=False
)

print("\n" + "=" * 80)
print("DATA CLEANING LOG CREATED")
print("=" * 80)

print(
    f"Cleaning actions logged: {len(cleaning_log_df)}"
)

print(
    "\nFile saved as: data_cleaning_log.csv"
)


DATA CLEANING LOG CREATED
Cleaning actions logged: 12

File saved as: data_cleaning_log.csv


In [ ]:
# ============================================================
# 13. CREATE CLEANED TABLE DICTIONARY
# ============================================================

cleaned_tables = {

    "Customers": customers,
    "Inventory": inventory,
    "Products": products,
    "Returns": returns,
    "Store Targets Legacy": store_targets_legacy,
    "Stores": stores,
    "Transactions": transactions,
    "Campaigns": campaigns

}

In [ ]:
# ============================================================
# 14. SAVE CLEANED DATA
# ============================================================

customers.to_csv(
    "cleaned_customers.csv",
    index=False
)

inventory.to_csv(
    "cleaned_inventory.csv",
    index=False
)

products.to_csv(
    "cleaned_products.csv",
    index=False
)

returns.to_csv(
    "cleaned_returns.csv",
    index=False
)

store_targets_legacy.to_csv(
    "cleaned_store_targets_legacy.csv",
    index=False
)

stores.to_csv(
    "cleaned_stores.csv",
    index=False
)

transactions.to_csv(
    "cleaned_transactions.csv",
    index=False
)

campaigns.to_csv(
    "cleaned_campaigns.csv",
    index=False
)


In [ ]:
# ============================================================
# 15. PROVE CLEANING WORKED
# ============================================================

print("\n" + "#" * 80)
print("MODULE 3 — AFTER-CLEANING VALIDATION")
print("#" * 80)


for name, df in cleaned_tables.items():

    print("\n" + "=" * 80)
    print(f"AFTER CLEANING — {name}")
    print("=" * 80)

    # Missing values
    missing_count = df.isna().sum().sum()

    print(
        f"Total missing values : {missing_count}"
    )

    # Duplicate rows
    duplicate_count = df.duplicated().sum()

    print(
        f"Duplicate rows       : {duplicate_count}"
    )

    # Text whitespace
    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    whitespace_issues = 0

    for column in text_columns:

        whitespace_issues += (
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
            .ne(
                df[column]
                .dropna()
                .astype(str)
            )
            .sum()
        )

    print(
        f"Whitespace issues    : {whitespace_issues}"
    )

    # Invalid dates
    date_columns = [
        column
        for column in df.columns
        if "date" in column.lower()
        or "month" in column.lower()
        or "period" in column.lower()
    ]

    invalid_dates = 0

    for column in date_columns:

        converted = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        invalid_dates += (
            df[column].notna()
            & converted.isna()
        ).sum()

    print(
        f"Invalid dates        : {invalid_dates}"
    )



################################################################################
MODULE 3 — AFTER-CLEANING VALIDATION
################################################################################

AFTER CLEANING — Customers
Total missing values : 63
Duplicate rows       : 22
Whitespace issues    : 0
Invalid dates        : 0

AFTER CLEANING — Inventory
Total missing values : 120
Duplicate rows       : 0
Whitespace issues    : 0
Invalid dates        : 0

AFTER CLEANING — Products
Total missing values : 19
Duplicate rows       : 8
Whitespace issues    : 0
Invalid dates        : 0

AFTER CLEANING — Returns
Total missing values : 10
Duplicate rows       : 15
Whitespace issues    : 0
Invalid dates        : 0

AFTER CLEANING — Store Targets Legacy
Total missing values : 0
Duplicate rows       : 0
Whitespace issues    : 0
Invalid dates        : 0

AFTER CLEANING — Stores
Total missing values : 2
Duplicate rows       : 0
Whitespace issues    : 0
Invalid dates        : 0

AFTER CLEANING — Tr

In [ ]:
# ============================================================
# 16. FINAL REFERENTIAL INTEGRITY PROOF
# ============================================================

print("\n" + "#" * 80)
print("FINAL REFERENTIAL INTEGRITY CHECK")
print("#" * 80)


# Transactions -> Customers
transaction_customer_check = (
    transactions["CustomerID"]
    .dropna()
    .astype(str)
    .str.strip()
    .isin(customer_ids)
)

print(
    "Transactions -> Customers:",
    "PASS" if transaction_customer_check.all() else "FAIL"
)


# Transactions -> Stores
transaction_store_check = (
    transactions["StoreID"]
    .dropna()
    .astype(str)
    .str.strip()
    .isin(store_ids)
)

print(
    "Transactions -> Stores:",
    "PASS" if transaction_store_check.all() else "FAIL"
)


# Transactions -> Products
transaction_product_check = (
    transactions["ProductID"]
    .dropna()
    .astype(str)
    .str.strip()
    .isin(product_ids)
)

print(
    "Transactions -> Products:",
    "PASS" if transaction_product_check.all() else "FAIL"
)


# Inventory -> Stores
inventory_store_check = (
    inventory["StoreID"]
    .dropna()
    .astype(str)
    .str.strip()
    .isin(store_ids)
)

print(
    "Inventory -> Stores:",
    "PASS" if inventory_store_check.all() else "FAIL"
)


# Inventory -> Products
inventory_product_check = (
    inventory["ProductID"]
    .dropna()
    .astype(str)
    .str.strip()
    .isin(product_ids)
)

print(
    "Inventory -> Products:",
    "PASS" if inventory_product_check.all() else "FAIL"
)


# Returns -> Transactions
return_transaction_check = (
    returns["TransactionID"]
    .dropna()
    .astype(str)
    .str.strip()
    .isin(transaction_ids)
)

print(
    "Returns -> Transactions:",
    "PASS" if return_transaction_check.all() else "FAIL"
)


# Store Targets -> Stores
target_store_check = (
    store_targets_legacy["store_code"]
    .dropna()
    .astype(str)
    .str.strip()
    .isin(store_ids)
)

print(
    "Store Targets -> Stores:",
    "PASS" if target_store_check.all() else "FAIL"
)


print("\n" + "#" * 80)
print("MODULE 3 CLEANING COMPLETED")
print("#" * 80)


################################################################################
FINAL REFERENTIAL INTEGRITY CHECK
################################################################################
Transactions -> Customers: PASS
Transactions -> Stores: FAIL
Transactions -> Products: FAIL
Inventory -> Stores: PASS
Inventory -> Products: PASS
Returns -> Transactions: PASS
Store Targets -> Stores: PASS

################################################################################
MODULE 3 CLEANING COMPLETED
################################################################################


In [ ]:
print("\nOrphan StoreID values:")
print(
    transaction_store_ids[
        orphan_transaction_stores
    ].value_counts().head(20)
)

print("\nOrphan ProductID values:")
print(
    transaction_product_ids[
        orphan_transaction_products
    ].value_counts().head(20)
)



print(products["ProductID"].tail(20).tolist())
print(products["ProductID"].nunique())


print(
    transactions[
        transactions["ProductID"].isin(
            orphan_product_values.index
        )
    ][["TransactionID", "ProductID"]].head(50)
)


Orphan StoreID values:
StoreID
ONLINE    27969
Name: count, dtype: int64

Orphan ProductID values:
ProductID
P9000    1
P9001    1
P9002    1
P9003    1
P9004    1
P9005    1
P9006    1
P9007    1
P9008    1
P9009    1
P9010    1
P9011    1
P9012    1
P9013    1
P9014    1
P9015    1
P9016    1
P9017    1
P9018    1
P9019    1
Name: count, dtype: int64
['P3509', 'P3510', 'P3511', 'P3512', 'P3513', 'P3514', 'P3515', 'P3516', 'P3517', 'P3518', 'P3519', 'P3520', 'P3406', 'P3405', 'P3416', 'P3155', 'P3263', 'P3324', 'P3118', 'P3269']
520


NameError: name 'orphan_product_values' is not defined

In [ ]:
# ============================================================
# INVESTIGATE ORPHAN PRODUCT IDs
# ============================================================

product_ids = set(
    products["ProductID"]
    .dropna()
    .astype(str)
    .str.strip()
)

transaction_product_ids = (
    transactions["ProductID"]
    .dropna()
    .astype(str)
    .str.strip()
)

orphan_product_mask = (
    ~transaction_product_ids.isin(product_ids)
)

orphan_product_values = (
    transaction_product_ids[
        orphan_product_mask
    ].value_counts()
)

print("\nOrphan ProductID values:")
print(orphan_product_values)

print("\nOrphan transaction records:")

print(
    transactions[
        transactions["ProductID"]
        .astype(str)
        .str.strip()
        .isin(orphan_product_values.index)
    ]
)


Orphan ProductID values:
ProductID
P9000    1
P9001    1
P9002    1
P9003    1
P9004    1
P9005    1
P9006    1
P9007    1
P9008    1
P9009    1
P9010    1
P9011    1
P9012    1
P9013    1
P9014    1
P9015    1
P9016    1
P9017    1
P9018    1
P9019    1
P9020    1
P9021    1
P9022    1
P9023    1
P9024    1
P9025    1
P9026    1
P9027    1
P9028    1
P9029    1
P9030    1
P9031    1
P9032    1
P9033    1
P9034    1
P9035    1
P9036    1
P9037    1
P9038    1
P9039    1
P9040    1
P9041    1
P9042    1
P9043    1
P9044    1
Name: count, dtype: int64

Orphan transaction records:
      TransactionID   OrderDate CustomerID StoreID ProductID  Quantity  \
60532      TX900000  2023-07-14    CU11185   ST105     P9000         1   
60533      TX900001  2024-09-30    CU10530   ST102     P9001         1   
60534      TX900002  2023-10-16    CU11648  ONLINE     P9002         2   
60535      TX900003  2024-10-10        NaN  ONLINE     P9003         1   
60536      TX900004  2023-01-08    CU12434  

In [ ]:
missing_transaction_customers = (
    transactions["CustomerID"].isna().sum()
)

print(
    f"Transactions with missing CustomerID: "
    f"{missing_transaction_customers}"
)


# ============================================================
# MISSING CUSTOMERID ANALYSIS
# ============================================================

missing_customer_transactions = transactions[
    transactions["CustomerID"].isna()
]

print("\nMissing CustomerID by Channel:")
print(
    missing_customer_transactions["Channel"]
    .value_counts(dropna=False)
)

print("\nMissing CustomerID by StoreID:")
print(
    missing_customer_transactions["StoreID"]
    .value_counts(dropna=False)
)

Transactions with missing CustomerID: 16874

Missing CustomerID by Channel:
Channel
Store     9108
Online    4531
App       3185
STORE       24
ONLINE      13
APP         13
Name: count, dtype: int64

Missing CustomerID by StoreID:
StoreID
ONLINE    7742
ST104      563
ST106      538
ST105      536
ST116      533
ST103      529
ST117      514
ST113      513
ST112      508
ST109      507
ST115      503
ST107      502
ST110      496
ST114      494
ST101      493
ST111      493
ST118      479
ST108      468
ST102      463
Name: count, dtype: int64


In [ ]:
print("\nMissing CustomerID + StoreID combination:")
print(
    missing_customer_transactions[
        ["StoreID", "Channel"]
    ]
    .value_counts()
)


Missing CustomerID + StoreID combination:
StoreID  Channel
ONLINE   Online     4531
         App        3185
ST104    Store       562
ST106    Store       537
ST105    Store       533
ST116    Store       530
ST103    Store       528
ST113    Store       513
ST117    Store       513
ST112    Store       508
ST109    Store       506
ST107    Store       500
ST115    Store       499
ST110    Store       496
ST114    Store       493
ST101    Store       493
ST111    Store       492
ST118    Store       478
ST108    Store       465
ST102    Store       462
ONLINE   ONLINE       13
         APP          13
ST115    STORE         4
ST116    STORE         3
ST105    STORE         3
ST108    STORE         3
ST107    STORE         2
ST102    STORE         1
ST109    STORE         1
ST106    STORE         1
ST103    STORE         1
ST104    STORE         1
ST114    STORE         1
ST111    STORE         1
ST117    STORE         1
ST118    STORE         1
Name: count, dtype: int64
